# DE Power Day-Ahead Fair Value Forecast

This notebook builds a **fair value model** for German (DE-LU) day-ahead power prices using publicly available data from the SMARD API and financial market feeds. The workflow covers:

1. **Data Ingestion** — Hourly generation forecasts (wind, solar, load) from SMARD + fuel prices (Henry Hub gas, Brent crude) from Yahoo Finance
2. **Data Validation** — Automated quality checks (completeness, physical bounds) before any cleaning
3. **Model Training** — A baseline Ridge regression vs. an improved HistGradientBoosting model with 20 selected features
4. **Signal Generation** — Hourly BUY/SELL/HOLD signals for the last available day, filtered by model reliability
5. **AI Diagnosis** — An LLM-powered analysis of model strengths, weaknesses, and improvement suggestions

---
**Data window:** Rolling 120-day lookback from today 
**Target variable:** `price_day_ahead_eur_mwh` (EPEX Spot DE-LU hourly auction) 
**Timezone:** Europe/Berlin (CET/CEST)

In [0]:
%pip install pandas_datareader -q

In [0]:
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import data_loader, data_validation, model_training, evaluate_day_ahead_forecast
importlib.reload(data_loader)
importlib.reload(data_validation)
importlib.reload(model_training)
importlib.reload(evaluate_day_ahead_forecast)

from data_loader import build_dataset, clean_dataset
from data_validation import validate_smard_series, validate_dataset, plot_data_quality_report
from model_training import run_fair_value_workflow
from evaluate_day_ahead_forecast import evaluate_day_ahead, display_trade_ideas

## Task 1: Data Ingestion & QA

The cell below performs three steps in sequence:

### Step 1: SMARD API Health Check
Verifies connectivity to all 6 SMARD time series endpoints (price, load, residual load, wind onshore/offshore, solar). Each must return HTTP 200.

### Step 2: Build Raw Dataset
`build_dataset()` fetches 120 days of hourly data from:
* **SMARD API** — 6 generation/load/price series for the DE-LU bidding zone
* **Yahoo Finance** (via `pandas_datareader`) — Henry Hub natural gas and Brent crude oil daily prices, forward-filled to hourly

Derived columns are added: `wind_total`, `vre_forecast`, `forecast_vre_share`, `forecast_wind_share`, `forecast_solar_share`.

### Step 3: Data Quality Report
`validate_dataset()` checks the raw (uninterpolated) data against predefined physical bounds and computes:
* **Completeness %** — fraction of non-null values per column
* **In-bounds %** — fraction of values within plausible physical ranges
* **Status** — OK (≥95%), Warning (80–95%), Critical (<80%)

### Step 4: Cleaning
`clean_dataset()` trims leading/trailing NaN rows (edge effects from API lag) and linearly interpolates interior gaps using time-weighted interpolation.

### Results Interpretation
* All SMARD series return **HTTP 200** (API healthy)
* Power/generation columns: **100% complete, 100% in-bounds** — no issues
* Fuel prices: **89% (gas) and 90% (Brent) complete** due to weekends/holidays with no market data — flagged as Warning but acceptable (interpolated in cleaning step)
* After cleaning: **2,689 rows** retained (191 edge rows dropped), covering Jan 5 – Apr 27, 2026

In [0]:
# Verify SMARD API availability
smard_status = validate_smard_series()
display(smard_status)

# Load raw dataset (120 days lookback)
df_raw = build_dataset()

# Validate raw data quality (before interpolation)
validation = validate_dataset(df_raw)
plot_data_quality_report(validation)

# Clean: drop edge NaN rows, interpolate interior gaps
df = clean_dataset(df_raw)
print(f"\nAfter cleaning: {df.shape[0]:,} rows (dropped {df_raw.shape[0] - df.shape[0]} edge rows, interpolated interior gaps)")
print(f"Date range: {df.index.min()} to {df.index.max()}")

## Task 2: Forecasting & Validation

Two models are trained on an **80/20 temporal split** (no shuffling — respects time ordering):

| Model | Algorithm | Features | Purpose |
| --- | --- | --- | --- |
| Baseline | Ridge Regression | 10 raw features (load, wind, solar, gas, oil, calendar) | Simple linear benchmark |
| Improved | HistGradientBoosting | 20 selected features | Production fair value model |

### Feature Selection (20 features from 91 candidates)
The improved model uses the **top 20 features** selected via permutation importance analysis on a held-out test set. This reduced complexity from 91 to 20 features with **no loss in accuracy** (MAE 20.9 vs 21.0, R² 0.704 vs 0.705).

The selected features fall into 5 categories:

| Category | Features | Importance |
| --- | --- | --- |
| Price-setting interaction | `gas_x_residual_positive` | Dominant (14.4) |
| Fundamentals | `residual_load`, `load`, `solar`, `vre`, `wind_total` | High (0.1–10.6) |
| Regime indicators | `scarcity_score`, `solar_x_surplus`, `vre_x_surplus_depth` | Medium (0.2–3.1) |
| Fuel prices | `gas_price`, `oil_price` | Medium (0.4–0.5) |
| Shares & calendar | `vre_share`, `solar_share`, `wind_share`, `residual_share`, hour/weekday encodings | Low–Medium (0.1–1.1) |

The single most important feature is `gas_x_residual_positive` — gas price multiplied by positive residual load — which captures the marginal cost of gas-fired generation during scarcity.

### Bias Correction
After training, systematic bias is removed by subtracting the mean training-set residual from all predictions.

### Model Comparison

The table below compares both models across 9 evaluation metrics on the held-out test set (538 hours). The **Delta** column shows the improvement of the Improved model over Baseline (negative = better for error metrics, positive = better for R²/correlation).

### Key Metrics Explained
| Metric | What it measures | Good value |
| --- | --- | --- |
| MAE | Average absolute error in EUR/MWh | Lower is better |
| RMSE | Penalizes large errors more than MAE | Lower is better |
| R² | Fraction of price variance explained | Closer to 1.0 |
| Bias | Systematic over/under-prediction | Closer to 0 |
| Peak MAE | Error during high-price hours (top 10%) | Lower means better peak capture |
| Low price MAE | Error during negative/low-price hours (bottom 10%) | Hardest to model well |
| Correlation | Linear agreement between predicted and actual | Closer to 1.0 |

### Results Interpretation
* The Improved model achieves **R² = 0.704** vs 0.693 for Baseline — a modest but consistent uplift
* **MAE:** 20.9 (Improved) vs 21.1 (Baseline) EUR/MWh — marginal improvement; the 20-feature model matches the full 91-feature model
* **Peak price MAE = 15.3 EUR/MWh** — the model handles high-demand spikes well
* **Low price MAE = 72.4 EUR/MWh** — extreme negative prices during renewable oversupply are the main weakness (both models struggle equally here)
* **Bias = +18.1 EUR/MWh** — the model systematically overpredicts, driven by inability to forecast deep negative prices
* **Correlation = 0.891** — strong directional agreement between predicted and actual prices



In [0]:
# Split: keep last 20% as test set
split_idx = int(len(df) * 0.8)
split_date = df.index[split_idx].strftime("%Y-%m-%d %H:%M")
# print(f"Train/test split at {split_date} (train: {split_idx:,} rows, test: {len(df) - split_idx:,} rows)")

# Run fair value models
output = run_fair_value_workflow(
    df,
    split_date=split_date,
    entry_threshold=10.0,
    bias_correct=True,
)

fair_value_results = output["results"]

# Model validation metrics
metric_descriptions = {
    "MAE": "Mean Absolute Error (EUR/MWh)",
    "RMSE": "Root Mean Squared Error (EUR/MWh)",
    "R2": "R-squared (explained variance)",
    "Bias_model_minus_actual": "Mean bias: model - actual (EUR/MWh)",
    "Spread_mean_actual_minus_fv": "Mean spread: actual - fair value (EUR/MWh)",
    "Spread_std": "Spread standard deviation (EUR/MWh)",
    "Peak_MAE_top_10pct": "MAE on peak prices (top 10%)",
    "Low_price_MAE_bottom_10pct": "MAE on low prices (bottom 10%)",
    "Correlation": "Pearson correlation (actual vs predicted)",
    "Train_bias_removed": "Training bias removed (EUR/MWh)",
}

metrics_df = pd.DataFrame({
    "Metric": [metric_descriptions.get(k, k) for k in output["baseline_metrics"]],
    "Baseline": list(output["baseline_metrics"].values()),
    "Improved": list(output["improved_metrics"].values()),
})
metrics_df["Delta"] = metrics_df["Improved"] - metrics_df["Baseline"]
metrics_df = metrics_df.round(3)

# Consistent table style with outlines
table_styles = [
    {"selector": "caption", "props": "font-size: 13px; font-weight: bold; text-align: left; padding: 6px 0;"},
    {"selector": "th", "props": "border: 1px solid #333; padding: 6px 10px; background-color: #f5f5f5; font-weight: bold;"},
    {"selector": "td", "props": "border: 1px solid #333; padding: 6px 10px;"},
    {"selector": "table", "props": "border-collapse: collapse; border: 2px solid #333;"},
]

display(
    metrics_df.style
    .hide(axis="index")
    .set_caption("Fair Value Model Comparison (Test Set)")
    .set_table_styles(table_styles)
    .format({"Baseline": "{:.3f}", "Improved": "{:.3f}", "Delta": "{:+.3f}"})
)

## Task 3: Prompt Curve Translation

Three charts provide visual insight into model behavior:

### Chart 1: Time Series — Actual vs Fair Value
Plots the actual day-ahead price against both model predictions over the test period. Look for:
* Periods where the improved model tracks actual prices well (most of the time)
* Spikes where both models miss — these are typically extreme negative price events

### Chart 2: Error vs Residual Load
Scatter plot of prediction error (fair value − actual) against residual load forecast. Key patterns:
* **Negative residual load** (renewable surplus) → large positive errors (model overpredicts)
* **Positive residual load** (conventional generation needed) → errors cluster near zero
* This confirms the model's weakness is in oversupply conditions

### Chart 3: Error vs Actual Price
Scatter plot of error against the actual price level:
* At **normal prices** (20–100 EUR/MWh): errors are small and symmetric
* At **negative prices** (-100 to -400 EUR/MWh): errors explode upward (model can't predict deep negatives)
* This directly motivates the error-threshold filtering used in the trade signal generation

In [0]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Time series: actual vs fair value
ax = axes[0]
ax.plot(fair_value_results.index, fair_value_results["actual_price"], alpha=0.7, label="Actual")
ax.plot(fair_value_results.index, fair_value_results["improved_fair_value"], alpha=0.7, label="Improved FV")
ax.plot(fair_value_results.index, fair_value_results["baseline_fair_value"], alpha=0.4, label="Baseline FV")
ax.set_ylabel("EUR/MWh")
ax.set_title("Day-Ahead Price vs Fair Value Models Test Set")
ax.legend()
ax.grid(True, alpha=0.3)

# Error vs residual load
ax = axes[1]
error = fair_value_results["improved_fair_value"] - fair_value_results["actual_price"]
x = df.loc[fair_value_results.index, "residual_load_forecast_mwh"]
ax.scatter(x, error, alpha=0.4, s=10)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Residual load forecast (MWh)")
ax.set_ylabel("Fair value error (EUR/MWh)")
ax.set_title("Improved Model Error vs Residual Load Forecast")
ax.grid(True, alpha=0.3)

# Error vs Price
ax = axes[2]
error = fair_value_results["improved_fair_value"] - fair_value_results["actual_price"]
x = df.loc[fair_value_results.index, "price_day_ahead_eur_mwh"]
ax.scatter(x, error, alpha=0.4, s=10)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Day-ahead price (EUR/MWh)")
ax.set_ylabel("Fair value error (EUR/MWh)")
ax.set_title("Improved Model Error vs Day Ahead Price Forecast")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

plt.tight_layout()
plt.show()


### Signal Logic (Dual-Condition Filtering)
A signal is only generated when **both** conditions are met:

1. **Spread condition:** |actual price − fair value| > `entry_threshold` (10 EUR/MWh)
2. **Reliability condition:** Historical model MAE at that price level ≤ `error_threshold` (30 EUR/MWh)

| Condition 1 | Condition 2 | Result |
| --- | --- | --- |
| Spread > threshold | MAE ≤ threshold | **BUY** or **SELL** (valid signal) |
| Spread > threshold | MAE > threshold | **SUPPRESSED** (model unreliable at this price) |
| Spread ≤ threshold | Any | **HOLD** (insufficient edge) |

### How Historical MAE is Computed
The test set prices are split into 10 equal-frequency bins. For each bin, the mean absolute error is computed. When evaluating a new hour, its price is mapped to the corresponding bin to look up reliability.

### Day Ahead forecast results
* **7 BUY signals** — market priced below fair value with reliable model at those price levels
* **0 SELL signals** — no hours where market was reliably priced above fair value
* **10 HOLD** — spread too small to trade
* **7 SUPPRESSED** — spread was large enough BUT model error exceeds 30 EUR/MWh at those price levels (deep negative prices during midday solar surplus)
* **Average spread: -69.61 EUR/MWh** — market broadly priced below fair value, directional bias is BUY
* The suppressed signals (visible as orange × markers in the chart) protect the trader from acting on unreliable predictions during extreme negative price hours (hours 10–16)

In [0]:
# Evaluate last full day and generate trade signals
result = evaluate_day_ahead(
    fair_value_results,
    df,
    entry_threshold=10.0,   # min spread to trigger a signal (EUR/MWh)
    error_threshold=30.0,   # max acceptable model error for that price level (EUR/MWh)
)

display_trade_ideas(result)

## Task 4: AI/LLM Integration

This cell uses **GPT-4o via the OpenAI API** to produce a natural-language interpretation of the model's performance, feature importance, and failure modes.

### What Gets Sent to the LLM
1. **Model metrics** — MAE (20.9), RMSE (40.0), R² (0.704), bias (+18.1), peak/low MAE, correlation
2. **Top 10 error-correlated features** — computed as the absolute Pearson correlation between each of the 20 model features and the prediction error on the test set
3. **Worst 5 forecast hours** — the hours with the largest absolute prediction errors

### How Feature-Error Correlation Works
Instead of relying on tree-based feature importances (which measure predictive contribution), we measure which features are most **associated with the model's mistakes**. A high correlation means that feature's values systematically co-occur with large errors — indicating the model hasn't learned that relationship well enough.

### Expected Results Interpretation
The LLM diagnosis typically identifies:
* **Solar-related features** (`solar_share`, `solar_x_surplus`, `solar`) dominate model errors — physically sensible since renewable oversupply drives extreme negative prices
* **VRE share and surplus depth** (`vre_share`, `vre_x_surplus_depth`) confirm the model breaks down during grid oversupply conditions
* **Worst hours** cluster around deep negative prices (-190 to -413 EUR/MWh) in April 2026, when solar production overwhelms demand
* **Suggestions** typically include: regime-switching model architecture, adding curtailment/grid constraint data, or quantile regression for tail events

> **Note:** Requires a valid `OPENAI_API_KEY` environment variable or Databricks secret scope.

In [0]:
from openai import OpenAI
from model_training import IMPROVED_FEATURES, create_fair_value_features, time_split

# Create openAI client
client = OpenAI(api_key='YOUR_API_KEY_HERE')

# Compute feature importances via correlation with prediction error
df_feat = create_fair_value_features(df)
df_feat = df_feat.dropna(subset=IMPROVED_FEATURES + ["target"])
_, test_feat = time_split(df_feat, split_date)

error = fair_value_results["improved_fair_value"] - fair_value_results["actual_price"]
importances = test_feat.loc[error.index, IMPROVED_FEATURES].corrwith(error).abs().sort_values(ascending=False)
top_features = importances.head(10)

# Worst forecast hours
worst_hours = fair_value_results.loc[
    (fair_value_results["improved_fair_value"] - fair_value_results["actual_price"]).abs().nlargest(5).index
]

# Build diagnostic prompt
prompt = f"""You are an energy trading analyst reviewing a day-ahead power price fair value model for the German (DE-LU) market.

Model Performance (test set, last 20% of data):
- MAE: {output['improved_metrics']['MAE']:.2f} EUR/MWh
- RMSE: {output['improved_metrics']['RMSE']:.2f} EUR/MWh
- R²: {output['improved_metrics']['R2']:.3f}
- Bias (model - actual): {output['improved_metrics']['Bias_model_minus_actual']:.2f} EUR/MWh
- Peak MAE (top 10%): {output['improved_metrics']['Peak_MAE_top_10pct']:.2f} EUR/MWh
- Low price MAE (bottom 10%): {output['improved_metrics']['Low_price_MAE_bottom_10pct']:.2f} EUR/MWh

Top 10 Features Most Correlated with Model Error (absolute correlation):
{top_features.to_string()}

Worst 5 Forecast Hours:
{worst_hours[['actual_price', 'improved_fair_value', 'improved_spread']].to_string()}

Please provide:
1. A brief diagnosis of model strengths and weaknesses
2. Which features drive the error and whether this makes physical sense
3. What price regimes the model struggles with and why
4. Two concrete suggestions to improve the model

Keep the response concise (under 300 words) and oriented toward a power trader."""

# Call OpenAI API

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=500,
)

diagnosis = response.choices[0].message.content
print("=" * 80)
print("AI MODEL DIAGNOSIS")
print("=" * 80)
print(diagnosis)